# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này triển khai hoàn chỉnh quy trình huấn luyện và đánh giá theo hướng dẫn.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt
!python -m pytest -q

In [ ]:
# Kaggle Secrets & Google Drive Mount
import os
import sys
from pathlib import Path

# Mount Google Drive để sao lưu checkpoints an toàn chống ngắt session Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/lab_day10_checkpoints'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Mounted Drive. Checkpoints will be backed up to: {DRIVE_DIR}')
except Exception as e:
    print('Not running on Colab or Drive mount skipped:', e)
    DRIVE_DIR = 'work/checkpoints'
    os.makedirs(DRIVE_DIR, exist_ok=True)

# Lấy Kaggle credentials từ Colab Secrets (KHÔNG ghi khoá vào notebook)
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print('Loaded Kaggle credentials from Colab Secrets.')
except Exception as e:
    print('Colab userdata not available or secret missing, checking environment:', e)

## 1. Dữ liệu: tải và chia split

In [ ]:
# Tải và chia dữ liệu (chia theo phiên quay session_key chống rò rỉ frame liền kề)
!python scripts/prepare_data.py --kaggle

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Đọc metadata và kiểm tra số lượng
with open('data/splits/meta.json', 'r', encoding='utf-8') as f:
    meta = json.load(f)
print('Dataset version:', meta['dataset_version'])
print('Counts per split:')
for split, cnt in meta['counts'].items():
    print(f'  {split:5s}: {cnt["images"]} images, boxes: {cnt["boxes"]}')

# Hàm vẽ bounding box lên ảnh kiểm tra (lưu vào report/images/gt_samples.png)
def visualize_samples(jsonl_path, num_samples=6, save_dir='report/images'):
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        records = [json.loads(line) for line in f]
    
    samples = [r for r in records if len(r['objects']) > 0][:num_samples]
    colors = {'pothole': 'red', 'crack': 'yellow', 'manhole': 'cyan'}
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    for idx, (rec, ax) in enumerate(zip(samples, axes)):
        img_path = Path('data') / rec['image_path']
        img = Image.open(img_path)
        ax.imshow(img)
        w, h = rec['width'], rec['height']
        
        for obj in rec['objects']:
            xc, yc, bw, bh = obj['yolo']
            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h
            rect_w = bw * w
            rect_h = bh * h
            label = obj['label']
            c = colors.get(label, 'lime')
            rect = patches.Rectangle((x1, y1), rect_w, rect_h, linewidth=2, edgecolor=c, facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, max(0, y1 - 5), label, color=c, fontsize=10, weight='bold',
                    bbox=dict(boxstyle='square,pad=0.1', fc='black', alpha=0.5, ec='none'))
        ax.set_title(f"{rec['image_id']} ({len(rec['objects'])} boxes)")
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(f'{save_dir}/gt_samples.png', dpi=150)
    plt.show()

visualize_samples('data/splits/train.jsonl', num_samples=6)

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
import json

PROMPT = (
    "Detect road damages in this dashcam image. "
    "Identify objects belonging to these classes: 'pothole', 'crack', 'manhole'. "
    "Return a JSON list of objects with 'label' and 'bbox_2d' on a 0-1000 scale: "
    '[{"label": "...", "bbox_2d": [x1, y1, x2, y2]}]. '
    "Do not output any additional explanation or markdown."
)

def yolo_to_box(label, yolo):
    """Chuyển toạ độ YOLO [xc, yc, w, h] chuẩn hoá [0,1] sang box [x1, y1, x2, y2] thang 0-1000."""
    xc, yc, w, h = yolo
    x1 = int(round(max(0.0, min(1000.0, (xc - w / 2) * 1000))))
    y1 = int(round(max(0.0, min(1000.0, (yc - h / 2) * 1000))))
    x2 = int(round(max(0.0, min(1000.0, (xc + w / 2) * 1000))))
    y2 = int(round(max(0.0, min(1000.0, (yc + h / 2) * 1000))))
    return {'label': label, 'bbox_2d': [x1, y1, x2, y2]}

def objects_to_answer(objects):
    """Chuyển danh sách objects sang chuỗi JSON rút gọn, sắp xếp theo thứ tự (y1, x1)."""
    boxes = [yolo_to_box(o['label'], o['yolo']) for o in objects]
    # Lọc box suy biến nếu x2 <= x1 hoặc y2 <= y1
    boxes = [b for b in boxes if b['bbox_2d'][2] > b['bbox_2d'][0] and b['bbox_2d'][3] > b['bbox_2d'][1]]
    # Sắp xếp trên -> dưới, trái -> phải: thứ tự ổn định cho model học
    boxes.sort(key=lambda b: (b['bbox_2d'][1], b['bbox_2d'][0]))
    return json.dumps(boxes, separators=(',', ':'))

# Kiểm tra thử trên 1 mẫu
sample_objs = [{'label': 'pothole', 'yolo': [0.5, 0.5, 0.2, 0.2]}]
print('Test objects_to_answer:', objects_to_answer(sample_objs))

## 3. E0: zero-shot

In [ ]:
import os
import torch
import gc
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from transformers import BitsAndBytesConfig, AutoProcessor
try:
    from transformers import Qwen3VLForConditionalGeneration
except ImportError:
    from transformers import AutoModelForImageTextToText as Qwen3VLForConditionalGeneration

MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'

# Cấu hình lượng tử 4-bit NF4, giữ ViT và lm_head ở fp16 (llm_int8_skip_modules)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=['visual', 'lm_head']
)

print('Loading processor & base model for E0...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = 'left'

model_e0 = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map={'': 0}
)
model_e0.eval()
print('Base model loaded successfully.')

# Hàm suy luận trên split
def run_inference(model, processor, split='test', out_jsonl='work/e0.jsonl', batch_size=1):
    Path(out_jsonl).parent.mkdir(parents=True, exist_ok=True)
    with open(f'data/splits/{split}.jsonl', 'r', encoding='utf-8') as f:
        records = [json.loads(line) for line in f]
    
    results = []
    print(f'Running inference on {len(records)} images from {split}...')
    
    for r in tqdm(records):
        img = Image.open(Path('data') / r['image_path']).convert('RGB')
        messages = [{
            'role': 'user',
            'content': [
                {'type': 'image', 'image': img},
                {'type': 'text', 'text': PROMPT}
            ]
        }]
        
        try:
            inputs = processor.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors='pt'
            )
        except Exception:
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = processor(images=img, text=text, return_dict=True, return_tensors='pt')
            
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
        prompt_len = inputs['input_ids'].shape[1]
        
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
            
        raw = processor.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
        results.append({'image_id': r['image_id'], 'raw': raw})
        
    with open(out_jsonl, 'w', encoding='utf-8') as f:
        for res in results:
            f.write(json.dumps(res, ensure_ascii=False) + '\n')
    print(f'Saved {len(results)} predictions to {out_jsonl}')

# Chạy suy luận Zero-shot E0 trên tập test
run_inference(model_e0, processor, split='test', out_jsonl='work/e0.jsonl')

# Dọn dẹp GPU giải phóng VRAM
del model_e0
gc.collect()
torch.cuda.empty_cache()
print('GPU memory cleared after E0.')

In [ ]:
# Chấm điểm E0
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class RoadDamageDataset(Dataset):
    def __init__(self, jsonl_path, img_dir='data'):
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            self.records = [json.loads(line) for line in f]
        self.img_dir = Path(img_dir)
        
    def __len__(self):
        return len(self.records)
        
    def __getitem__(self, idx):
        rec = self.records[idx]
        img_path = self.img_dir / rec['image_path']
        img = Image.open(img_path).convert('RGB')
        answer = objects_to_answer(rec['objects'])
        return img, PROMPT, answer, rec['image_id']

def encode_example(image, prompt, answer, processor):
    user_msg = {'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}]}
    asst_msg = {'role': 'assistant', 'content': [{'type': 'text', 'text': answer}]}
    
    try:
        prompt_enc = processor.apply_chat_template(
            [user_msg], add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt'
        )
        full_enc = processor.apply_chat_template(
            [user_msg, asst_msg], add_generation_prompt=False, tokenize=True, return_dict=True, return_tensors='pt'
        )
    except Exception:
        text_prompt = processor.apply_chat_template([user_msg], tokenize=False, add_generation_prompt=True)
        text_full = processor.apply_chat_template([user_msg, asst_msg], tokenize=False, add_generation_prompt=False)
        prompt_enc = processor(images=image, text=text_prompt, return_dict=True, return_tensors='pt')
        full_enc = processor(images=image, text=text_full, return_dict=True, return_tensors='pt')
    
    n = prompt_enc['input_ids'].shape[1]
    assert torch.equal(full_enc['input_ids'][0, :n], prompt_enc['input_ids'][0]), \
        'Prompt token prefix mismatch!'
        
    labels = full_enc['input_ids'].clone()
    labels[0, :n] = -100  # Mask loss: Chỉ học phần trả lời JSON + <|im_end|>
    
    item = {k: v.squeeze(0) for k, v in full_enc.items()}
    item['labels'] = labels.squeeze(0)
    return item

def collate_fn(batch_items, processor):
    pad_id = processor.tokenizer.pad_token_id if processor.tokenizer.pad_token_id is not None else 0
    seq_keys = ['input_ids', 'attention_mask', 'labels']
    if 'mm_token_type_ids' in batch_items[0]:
        seq_keys.append('mm_token_type_ids')
        
    max_len = max(item['input_ids'].shape[0] for item in batch_items)
    
    out = {}
    for key in seq_keys:
        pad_val = -100 if key == 'labels' else (0 if key != 'input_ids' else pad_id)
        padded = []
        for item in batch_items:
            t = item[key]
            diff = max_len - t.shape[0]
            if diff > 0:
                pad_t = torch.full((diff,), pad_val, dtype=t.dtype)
                padded.append(torch.cat([t, pad_t], dim=0))
            else:
                padded.append(t)
        out[key] = torch.stack(padded, dim=0)
        
    # pixel_values và image_grid_thw nối theo chiều 0 (cat), KHÔNG pad
    for vkey in ['pixel_values', 'image_grid_thw']:
        if vkey in batch_items[0]:
            out[vkey] = torch.cat([it[vkey] if it[vkey].dim() > 1 else it[vkey].unsqueeze(0) for it in batch_items], dim=0)
            
    return out

# Kiểm tra thử encode_example và in unmasked tokens
train_ds = RoadDamageDataset('data/splits/train.jsonl')
img_0, p_0, a_0, _ = train_ds[0]
ex_0 = encode_example(img_0, p_0, a_0, processor)
print('Input IDs shape:', ex_0['input_ids'].shape)
print('Labels shape:', ex_0['labels'].shape)
unmasked_tokens = ex_0['input_ids'][ex_0['labels'] != -100]
print('Decoded answer from unmasked labels:', processor.decode(unmasked_tokens, skip_special_tokens=False))

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
import re
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup_type='E1', lora_r=16, lora_alpha=32, lora_dropout=0.05):
    """
    Cấu hình LoRA và đóng băng/mở khoá các khối theo đúng GUIDE:
    - E1: LoRA chỉ trên LLM (252 modules), đóng băng ViT & Projector.
    - E2: E1 + mở đầy đủ Projector (visual.merger và visual.deepstack_merger_list, float32, ~109M params).
    - E3: E2 + LoRA trên ViT (thêm 96 modules -> tổng 348 LoRA modules).
    """
    # 1. Đóng băng tất cả tham số
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()
    
    # 2. Xác định target_modules LoRA
    if setup_type in ('E1', 'E2'):
        target_regex = r'.*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)$'
    elif setup_type == 'E3':
        target_regex = (
            r'.*(language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)|'
            r'visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2)))$'
        )
    else:
        raise ValueError(f'Unknown setup_type: {setup_type}')
        
    peft_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=target_regex,
        bias='none',
        task_type='CAUSAL_LM'
    )
    peft_model = get_peft_model(model, peft_config)
    
    # 3. Mở Projector nếu là E2 hoặc E3 (đưa về float32 và requires_grad=True)
    if setup_type in ('E2', 'E3'):
        for name, module in peft_model.named_modules():
            if any(k in name for k in ('visual.merger', 'visual.deepstack_merger_list')):
                module.to(torch.float32)
                for p in module.parameters():
                    p.requires_grad = True
        for name, param in peft_model.named_parameters():
            if 'visual.merger' in name or 'visual.deepstack_merger_list' in name:
                param.data = param.data.to(torch.float32)
                param.requires_grad = True
                
    # 4. Đảm bảo 100% tất cả tham số trainable đều ở float32 (tránh lỗi GradScaler unscale FP16)
    for name, param in peft_model.named_parameters():
        if param.requires_grad:
            param.data = param.data.float()
            
    # Đếm số modules khớp LoRA và tổng tham số trainable
    lora_modules = [n for n, m in peft_model.named_modules() if hasattr(m, 'lora_A')]
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in peft_model.parameters())
    
    print(f'[{setup_type}] LoRA target modules count: {len(lora_modules)}')
    print(f'[{setup_type}] Trainable params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)')
    
    return peft_model

## 6. Vòng huấn luyện

In [ ]:
import time
import shutil
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

def train_experiment(setup_type='E1', lora_r=16, lora_alpha=32, max_steps=300, lr=2e-4, batch_size=2, grad_accum=4):
    print(f'\n================ STARTING EXPERIMENT {setup_type} (r={lora_r}, alpha={lora_alpha}) ================')
    # 1. Khởi tạo lại base model ở 4-bit
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={'': 0}
    )
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
    
    # 2. Áp dụng setup
    model = apply_setup(model, setup_type=setup_type, lora_r=lora_r, lora_alpha=lora_alpha)
    model.train()
    
    # 3. Dataloader
    dataset = RoadDamageDataset('data/splits/train.jsonl')
    
    def collate_wrapper(batch):
        items = [encode_example(img, p, ans, processor) for img, p, ans, _ in batch]
        return collate_fn(items, processor)
        
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_wrapper)
    data_iter = iter(dataloader)
    
    # 4. Optimizer & Scheduler
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=lr, weight_decay=0.0)
    warmup_steps = int(0.05 * max_steps)
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=max_steps)
    scaler = torch.cuda.amp.GradScaler()
    
    # 5. Tracking
    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()
    save_dir = f'work/{setup_type.lower()}'
    os.makedirs(save_dir, exist_ok=True)
    
    # 6. Training loop
    for step in range(1, max_steps + 1):
        optimizer.zero_grad()
        accum_loss = 0.0
        
        for _ in range(grad_accum):
            try:
                batch = next(data_iter)
            except StopIteration:
                data_iter = iter(dataloader)
                batch = next(data_iter)
                
            batch = {k: v.to('cuda') for k, v in batch.items()}
            with torch.cuda.amp.autocast(dtype=torch.float16):
                loss = model(**batch).loss / grad_accum
                
            if not torch.isfinite(loss):
                raise RuntimeError(f'Loss is NaN or Inf at step {step}!')
                
            scaler.scale(loss).backward()
            accum_loss += loss.item() * grad_accum
            
        # Chuyển gradient về float32 nếu có trước khi unscale (chống lỗi GradScaler)
        for p in trainable_params:
            if p.grad is not None and p.grad.dtype == torch.float16:
                p.grad = p.grad.float()
                
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        if step % 25 == 0 or step == max_steps:
            current_lr = scheduler.get_last_lr()[0]
            print(f'Step {step:3d}/{max_steps} | Loss: {accum_loss:.4f} | LR: {current_lr:.6f}')
            
    # Kết thúc train
    elapsed_minutes = (time.time() - start_time) / 60.0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
    num_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f'\nTraining completed in {elapsed_minutes:.2f} mins.')
    print(f'Peak VRAM: {peak_vram_gb:.2f} GB | Trainable params: {num_trainable:,}')
    
    # 7. Lưu adapter LoRA
    model.save_pretrained(save_dir)
    
    # Lưu riêng trọng số Projector nếu là E2 hoặc E3 (đặc biệt quan trọng!)
    if setup_type in ('E2', 'E3'):
        proj_sd = {
            k: v.detach().cpu() for k, v in model.named_parameters()
            if ('visual.merger' in k or 'visual.deepstack_merger_list' in k) and v.requires_grad
        }
        torch.save(proj_sd, os.path.join(save_dir, 'projector.pt'))
        print(f'Saved {len(proj_sd)} projector tensors to {save_dir}/projector.pt')
        
    # Ghi run.json
    run_meta = {
        'trainable_params': int(num_trainable),
        'peak_vram_gb': round(float(peak_vram_gb), 2),
        'train_minutes': round(float(elapsed_minutes), 2)
    }
    with open(os.path.join(save_dir, 'run.json'), 'w', encoding='utf-8') as f:
        json.dump(run_meta, f, indent=2)
        
    # Sao lưu sang Google Drive
    try:
        drive_dest = os.path.join(DRIVE_DIR, setup_type.lower())
        shutil.copytree(save_dir, drive_dest, dirs_exist_ok=True)
        print(f'Backed up {setup_type} adapter to Google Drive: {drive_dest}')
    except Exception as e:
        print('Backup to Drive failed:', e)
        
    # Dọn dẹp GPU
    del model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()
    
    return run_meta

## 7. Suy luận và chấm E1–E3

In [ ]:
from peft import PeftModel

def evaluate_and_score(setup_type='E1', split='test', custom_save_dir=None, custom_name=None, custom_out=None):
    save_dir = custom_save_dir or f'work/{setup_type.lower()}'
    exp_name = custom_name or setup_type
    out_evidence = custom_out or f'report/evidence/{setup_type.lower()}.json'
    
    with open(os.path.join(save_dir, 'run.json'), 'r', encoding='utf-8') as f:
        meta_info = json.load(f)
        
    print(f'\n================ EVALUATING {exp_name} ================')
    # 1. Load base model
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map={'': 0}
    )
    
    # 2. Load LoRA adapter
    model = PeftModel.from_pretrained(model, save_dir)
    
    # 3. Load Projector weights nếu là E2 hoặc E3
    proj_path = os.path.join(save_dir, 'projector.pt')
    if os.path.exists(proj_path):
        proj_sd = torch.load(proj_path, map_location='cuda:0')
        model.load_state_dict(proj_sd, strict=False)
        print(f'Successfully loaded {len(proj_sd)} projector tensors from {proj_path}')
        
    model.eval()
    
    # 4. Suy luận
    out_jsonl = f'work/{exp_name.lower()}.jsonl'
    run_inference(model, processor, split=split, out_jsonl=out_jsonl, batch_size=1)
    
    # Dọn dẹp GPU
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    # 5. Chấm điểm với scripts/score.py
    cmd = (
        f'python scripts/score.py --pred {out_jsonl} --name {exp_name} --out {out_evidence} '
        f'--meta trainable_params={meta_info["trainable_params"]} '
        f'--meta peak_vram_gb={meta_info["peak_vram_gb"]} '
        f'--meta train_minutes={meta_info["train_minutes"]}'
    )
    print('Running score command:', cmd)
    os.system(cmd)

In [ ]:
# Huấn luyện và chấm điểm tuần tự E1, E2, E3
for exp in ['E1', 'E2', 'E3']:
    train_experiment(setup_type=exp)
    evaluate_and_score(setup_type=exp)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# ==================== E4: ABLATION STUDY (RANK r=4 vs r=64) ====================
# Khảo sát biến rank trên cấu hình E1: r=4 (alpha=8) và r=64 (alpha=128)
print('=== Running E4: Rank r=4 ===')
meta_r4 = train_experiment(setup_type='E1', lora_r=4, lora_alpha=8)
shutil.copytree('work/e1', 'work/e4_rank_4', dirs_exist_ok=True)
evaluate_and_score(
    setup_type='E1',
    custom_save_dir='work/e4_rank_4',
    custom_name='E4_rank_4',
    custom_out='report/evidence/e4_rank_4.json'
)

print('=== Running E4: Rank r=64 ===')
meta_r64 = train_experiment(setup_type='E1', lora_r=64, lora_alpha=128)
shutil.copytree('work/e1', 'work/e4_rank_64', dirs_exist_ok=True)
evaluate_and_score(
    setup_type='E1',
    custom_save_dir='work/e4_rank_64',
    custom_name='E4_rank_64',
    custom_out='report/evidence/e4_rank_64.json'
)

In [ ]:
# ==================== E5: BONUS GRPO REWARD FUNCTION & TEST ====================
from scripts.score import parse_output, iou_xyxy

def compute_iou_reward(completion, gt_boxes):
    """
    Hàm tính phần thưởng IoU cho GRPO (GUIDE bước 9):
    - Trả về 0.0 nếu JSON hỏng
    - Trả về 1.0 nếu cả GT và Pred đều rỗng
    - Ghép đôi tham lam theo IoU giảm dần, cùng nhãn, IoU >= 0.5
    - Công thức: 0.2 + 0.8 * sum(iou) / max(len(gt), len(pred), 1)
    """
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if not gt_boxes and not pred:
        return 1.0
        
    pairs = []
    for i, g in enumerate(gt_boxes):
        for j, p in enumerate(pred):
            if g['label'] == p['label']:
                val = iou_xyxy(g['bbox'], p['bbox'])
                if val >= 0.5:
                    pairs.append((val, i, j))
    pairs.sort(reverse=True)
    
    used_g, used_p, total_iou = set(), set(), 0.0
    for val, i, j in pairs:
        if i in used_g or j in used_p:
            continue
        used_g.add(i)
        used_p.add(j)
        total_iou += val
        
    denom = max(len(gt_boxes), len(pred), 1)
    reward = 0.2 + 0.8 * (total_iou / denom)
    return float(reward)

# Kiểm tra các trường hợp biên của hàm thưởng
gt_test = [{'label': 'pothole', 'bbox': (400, 400, 600, 600)}]
assert compute_iou_reward(None, gt_test) == 0.0, 'None input failed'
assert compute_iou_reward('invalid json', gt_test) == 0.0, 'Corrupt json failed'
assert compute_iou_reward('[]', []) == 1.0, 'Both empty failed'
# Hoàn hảo
assert compute_iou_reward('[{"label": "pothole", "bbox_2d": [400, 400, 600, 600]}]', gt_test) == 1.0, 'Perfect failed'
# Sai nhãn
assert compute_iou_reward('[{"label": "crack", "bbox_2d": [400, 400, 600, 600]}]', gt_test) == 0.2, 'Wrong class failed'
print('Reward function tests passed successfully!')

In [ ]:
# ==================== TRỰC QUAN HOÁ 6 ẢNH SO SÁNH PREDICTION VS GROUND TRUTH ====================
# Phục vụ tiêu chí B (RUBRIC: >= 6 ảnh có chú thích Ground Truth vs Prediction)
def plot_prediction_comparisons(pred_file='work/e1.jsonl', split_file='data/splits/test.jsonl', out_dir='report/images'):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    with open(split_file, 'r', encoding='utf-8') as f:
        gts = {r['image_id']: r for r in (json.loads(l) for l in f)}
    with open(pred_file, 'r', encoding='utf-8') as f:
        preds = {r['image_id']: r['raw'] for r in (json.loads(l) for l in f)}
        
    colors_gt = {'pothole': 'lime', 'crack': 'cyan', 'manhole': 'yellow'}
    colors_pr = {'pothole': 'red', 'crack': 'orange', 'manhole': 'magenta'}
    
    # Chọn 6 ảnh có object để vẽ
    sample_ids = [k for k, v in gts.items() if len(v['objects']) > 0][:6]
    
    for idx, sid in enumerate(sample_ids):
        rec = gts[sid]
        raw_pred = preds.get(sid, '[]')
        pred_boxes = parse_output(raw_pred) or []
        
        img = Image.open(Path('data') / rec['image_path']).convert('RGB')
        w, h = rec['width'], rec['height']
        
        plt.figure(figsize=(10, 6))
        ax = plt.gca()
        ax.imshow(img)
        
        # Vẽ Ground Truth (nét liền)
        for obj in rec['objects']:
            xc, yc, bw, bh = obj['yolo']
            x1, y1 = (xc - bw/2) * w, (yc - bh/2) * h
            rect_w, rect_h = bw * w, bh * h
            lab = obj['label']
            rect = patches.Rectangle((x1, y1), rect_w, rect_h, linewidth=2.5,
                                     edgecolor=colors_gt.get(lab, 'lime'), facecolor='none', linestyle='-')
            ax.add_patch(rect)
            ax.text(x1, max(0, y1 - 4), f'GT: {lab}', color=colors_gt.get(lab, 'lime'),
                    fontsize=9, weight='bold', bbox=dict(boxstyle='square,pad=0.1', fc='black', alpha=0.6, ec='none'))
                    
        # Vẽ Prediction (nét đứt)
        for pb in pred_boxes:
            px1, py1, px2, py2 = pb['bbox']
            x1, y1 = px1 / 1000 * w, py1 / 1000 * h
            rect_w, rect_h = (px2 - px1) / 1000 * w, (py2 - py1) / 1000 * h
            lab = pb['label']
            rect = patches.Rectangle((x1, y1), rect_w, rect_h, linewidth=2,
                                     edgecolor=colors_pr.get(lab, 'red'), facecolor='none', linestyle='--')
            ax.add_patch(rect)
            ax.text(x1, min(h - 5, y1 + rect_h + 12), f'Pred: {lab}', color=colors_pr.get(lab, 'red'),
                    fontsize=9, weight='bold', bbox=dict(boxstyle='square,pad=0.1', fc='black', alpha=0.6, ec='none'))
                    
        plt.title(f'Comparison {idx+1}: {sid} (Solid: Ground Truth, Dashed: Prediction)')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig(f'{out_dir}/comparison_{idx+1}.png', dpi=150)
        plt.close()
        
    print(f'Generated 6 comparison images in {out_dir}')

plot_prediction_comparisons()

In [ ]:
# Kiểm tra các điều kiện nộp bài trước khi kết thúc
!python scripts/check_submission.py